# Compute Model Perplexity on Generated Predictions

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, GenerationConfig, EncoderDecoderModel
from tqdm import tqdm
from datasets import load_dataset

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device.upper()}")

Using device: CUDA


In [7]:
cache_dir = "/mimer/NOBACKUP/groups/naiss2023-6-290/stefano/hf_cache/"
ds = load_dataset(
    'ailab-bio/PROTAC-Splitter-Dataset',
    'clustered',
    cache_dir=cache_dir,
)
test_ds = ds['held_out']

In [ ]:
checkpoint_dir = '/mimer/NOBACKUP/groups/naiss2023-6-290/stefano/models/'
model_name = checkpoint_dir + "PROTAC-Splitter-EncoderDecoder-lr_reduce-rand-smiles"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = EncoderDecoderModel.from_pretrained(model_name)
model.eval()
model = model.to(device)

In [ ]:
GENERATION_STRATEGY_PARAMS = {
    "greedy": {"num_beams": 1, "do_sample": False},
    "contrastive_search": {"penalty_alpha": 0.1, "top_k": 10},
    "multinomial_sampling": {"num_beams": 1, "do_sample": True},
    "beam_search_decoding": {"num_beams": 5, "do_sample": False, "num_return_sequences": 1},
    "beam_search_multinomial_sampling": {"num_beams": 5, "do_sample": True, "num_return_sequences": 1},
    "diverse_beam_search_decoding": {"num_beams": 5, "num_beam_groups": 5, "diversity_penalty": 1.0, "num_return_sequences": 1},
}

def get_generation_config(generation_strategy: str) -> GenerationConfig:
    """ Get the generation config for the given generation strategy. """
    return GenerationConfig(
        max_length=512,
        max_new_tokens=512,
        **GENERATION_STRATEGY_PARAMS[generation_strategy],
    )

batch_size = 16

# Step 1: Encode input
inputs = tokenizer(
    test_ds['text'][:batch_size],  # batch size = 1
    return_tensors='pt',
    padding='max_length',
    truncation=True,
    max_length=512,
).to(device)

# Step 2: Generate output
# TODO: Pre-load a generation config to use
generation_config = get_generation_config('beam_search_multinomial_sampling')
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        generation_config=generation_config,
    )

print(generated_ids.shape)

# Step 3: Use generated output directly
decoder_input_ids = generated_ids
labels = decoder_input_ids.clone()
labels[labels == tokenizer.pad_token_id] = -100
# Generate decoder attention mask
decoder_attention_mask = torch.ones_like(decoder_input_ids)
decoder_attention_mask[decoder_input_ids == tokenizer.pad_token_id] = 0

# Step 4: Compute loss on generated output
# NOTE: Since we have an encoder-decoder model, the "inputs" are actually the
# ones to the encoder. We however need the logits on the decoder outputs, so
# we need to input them accordingly.
with torch.no_grad():
    output = model(
        **inputs,
        decoder_input_ids=decoder_input_ids,
        decoder_attention_mask=decoder_attention_mask,
        labels=labels,
    )

# Step 5: Compute perplexity
loss = output.loss  # this is averaged over non-masked tokens
perplexity = torch.exp(loss)

print(f"Generated: {tokenizer.decode(generated_ids[0], skip_special_tokens=True)}")
print(f"Perplexity on model's own prediction: {perplexity.item():,.4f}")

torch.Size([16, 173])
Generated: Cc1ncsc1-c1ccc([C@H](C)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](N[*:2])C(C)(C)C)cc1.O=C(CC(=O)[*:2])[*:1].CC1(C)CCC(c2ccc(Cl)cc2)=C(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(N[C@H](CCN5CCN([*:1])CC5)CSc5ccccc5)c(S(=O)(=O)C(F)(F)F)c4)cc3)CC2)C1
Perplexity on model's own prediction: 8,567,326,208.0000


In [45]:
logits = output.logits
print(f"Logits shape: {logits.shape}")
print(f"Inputs shape: {decoder_input_ids.shape}")

Logits shape: torch.Size([16, 173, 767])
Inputs shape: torch.Size([16, 173])


Check [Perplexity from torchmetrics](https://lightning.ai/docs/torchmetrics/stable/gallery/text/perplexity.html) for more information.

In [ ]:
from torchmetrics.text import Perplexity

for l, generated_target, text in zip(logits, decoder_input_ids, test_ds['text'][:batch_size]):
    l = l.unsqueeze(0)
    decoded_target = tokenizer.decode(generated_target, skip_special_tokens=True)
    generated_target = generated_target.unsqueeze(0)

    print(f"Input:     {text}")
    print(f"Generated: {decoded_target}")
    print(f"Equal: {text == decoded_target}")

    perplexity = Perplexity().to(device)
    score = perplexity(preds=l, target=generated_target)
    print(f"Perplexity, unshifted: {score.item():,}")

    perplexity = Perplexity(ignore_index=None).to(device)
    score = perplexity(preds=l[:, :-1], target=generated_target[:, 1:])
    print(f"Perplexity, including padding: {score.item()}")

    perplexity = Perplexity(ignore_index=tokenizer.pad_token_id).to(device)
    score = perplexity(preds=l[:, :-1], target=generated_target[:, 1:])
    print(f"Perplexity, ignoring padding:  {score.item()}")
    print('-' * 80)

# NOTE: Perplexity can be calculated on the whole batch, I think.
# perplexity = Perplexity().to(device)
# score = perplexity(preds=logits, target=decoder_input_ids)
# print(f"Perplexity, unshifted: {score.item():,}")

# perplexity = Perplexity(ignore_index=None).to(device)
# score = perplexity(preds=logits[:, :-1], target=decoder_input_ids[:, 1:])
# print(f"Perplexity, including padding: {score.item()}")

# perplexity = Perplexity(ignore_index=tokenizer.pad_token_id).to(device)
# score = perplexity(preds=logits[:, :-1], target=decoder_input_ids[:, 1:])
# print(f"Perplexity, ignoring padding:  {score.item()}")

Input:     Cc1ncsc1-c1ccc([C@H](C)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)CC(=O)N2CCN(CC[C@H](CSc3ccccc3)Nc3ccc(S(=O)(=O)NC(=O)c4ccc(N5CCN(CC6=C(c7ccc(Cl)cc7)CCC(C)(C)C6)CC5)cc4)cc3S(=O)(=O)C(F)(F)F)CC2)C(C)(C)C)cc1
Generated: Cc1ncsc1-c1ccc([C@H](C)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](N[*:2])C(C)(C)C)cc1.O=C(CC(=O)[*:2])[*:1].CC1(C)CCC(c2ccc(Cl)cc2)=C(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(N[C@H](CCN5CCN([*:1])CC5)CSc5ccccc5)c(S(=O)(=O)C(F)(F)F)c4)cc3)CC2)C1
Equal: False
Perplexity, unshifted: 8,416,631,808.0
Perplexity, including padding: 1.2078497409820557
Perplexity, ignoring padding:  1.0000001192092896
--------------------------------------------------------------------------------
Input:     Cc1ncsc1-c1ccc([C@H](C)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)CCC(=O)N2CCN(CC[C@H](CSc3ccccc3)Nc3ccc(S(=O)(=O)NC(=O)c4ccc(N5CCN(CC6=C(c7ccc(Cl)cc7)CCC(C)(C)C6)CC5)cc4)cc3S(=O)(=O)C(F)(F)F)CC2)C(C)(C)C)cc1
Generated: Cc1ncsc1-c1ccc([C@H](C)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](N[*:2])C(C